# Part 2. Data Quality and Clean Bank–Quarter Panel

## Objective and dependency

The input is the verified FDIC acquisition generated in **Part 1 / Notebook 01**. The output is (i) an automated quality audit, (ii) a chronological clean bank–quarter panel, and (iii) a separate retrospective exit reconciliation.

**Execution:** 1. Load repository and validate tests → 2. Confirm Part 1 source → 3. Define documented quality policy and run ID → 4. Audit and construct the panel → 5. Inspect quantitative quality results → 6. Evaluate exit classification and information timing → 7. Revalidate lineage and hand off to Part 3.

The notebook never calls the FDIC API, changes acquired files, creates a failure label, or joins retrospective exit classification to the point-in-time predictor panel.


## 1. Repository, dependencies and Part 2 tests

The runtime uses the same pinned dependency file as Part 1. The Part 2 tests cover missing values, logical anomaly flags, chronological gaps, failure/exit distinctions, cross-stage SHA-256 checks and real Parquet output on synthetic data. The integration tests must run without skipped cases before the FDIC data are processed.


In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile
import xml.etree.ElementTree as ET

REPOSITORY = "https://github.com/Duongnb58/BFT65B_11230945_Final.git"
REPO_DIR = Path(tempfile.mkdtemp(prefix="bft65b_part2_", dir="/content")) / "repository"
subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(REPO_DIR)], check=True)

required_files = (
    "requirements.txt",
    "src/fdic_pipeline/acquisition.py",
    "src/fdic_pipeline/workflow.py",
    "src/fdic_pipeline/data_quality.py",
    "src/fdic_pipeline/panel.py",
    "run_data_quality.py",
    "tests/test_data_quality.py",
    "tests/test_panel_integration.py",
)
missing = [name for name in required_files if not (REPO_DIR / name).is_file()]
assert not missing, f"Missing Part 2 files in repository: {missing}"
COMMIT_SHA = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(REPO_DIR / "requirements.txt")], check=True)

TEST_XML = Path("/content/bft65b_part2_pytest.xml")
subprocess.run([
    sys.executable, "-m", "pytest", "-q", "-rs",
    "--junitxml", str(TEST_XML),
    str(REPO_DIR / "tests" / "test_data_quality.py"),
    str(REPO_DIR / "tests" / "test_panel_integration.py"),
], cwd=REPO_DIR, check=True)
suites = ET.parse(TEST_XML).getroot().findall(".//testsuite")
assert suites, "Missing machine-readable test report"
assert sum(int(s.get("skipped", 0)) for s in suites) == 0, "Integration tests were skipped"
assert sum(int(s.get("failures", 0)) + int(s.get("errors", 0)) for s in suites) == 0
print("Part 2 tests: PASSED; skipped: 0")
print("Source-code revision:", COMMIT_SHA)
print("Source directory:", REPO_DIR)


## 2. Verified Part 1 input

The data-quality stage reads **only** the immutable acquisition directory. Its entry contract checks the calendar, source field lists, quarter inventory, reference tables, number of records, and hashes of the manifest and verification report. During execution, the pipeline additionally checks SHA-256, schema, and row counts of each source Parquet file.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import sys
sys.path.insert(0, str(REPO_DIR / "src"))
from fdic_pipeline.panel import validate_source

DRIVE_ROOT = Path("/content/drive/MyDrive/BFT65B_11230945_Final")
SOURCE_DIR = DRIVE_ROOT / "fdic_1984Q1_2026Q2"
ACQUISITION, SOURCE_MANIFEST = validate_source(SOURCE_DIR)
assert ACQUISITION["quarters"] == 170
assert ACQUISITION["financial_fields"] == 33
print("Part 1 acquisition: PASSED")
print("Financials records:", f"{ACQUISITION['financial_rows']:,}")
print("Distinct CERT:", f"{ACQUISITION['distinct_financial_certs']:,}")
print("Failures / without CERT:", ACQUISITION["failures_rows"],
      "/", ACQUISITION["failures_missing_cert_rows"])
print("Institutions:", ACQUISITION["institutions_rows"])
print("Source manifest SHA-256:", ACQUISITION["manifest_sha256"])
print("Source verification SHA-256:", ACQUISITION["verification_report_sha256"])


## 3. Data-quality policy and output identity

**Missingness:** report by field and reporting year; retain missing values. **Core validity:** quarantine rows with missing or nonpositive `ASSET` and preserve their bank–quarter keys for review. **Screening:** flag negative deposits/loans, component totals above assets and extreme percentage ratios; do not overwrite source values. **Panel history:** calculate prior-report count and gaps from earlier reports only; do not fill non-reporting quarters.

`ASSUMED_AVAILABLE_FROM` is the reporting date plus a declared 90-day planning lag, **not an observed publication timestamp**. Full historical publication-vintage integrity cannot be established from the Part 1 extract alone. Future failure events and institutional terminal status stay outside the predictor panel.


In [ ]:
from datetime import datetime, timezone

AVAILABILITY_LAG_DAYS = 90
RUN_ID = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
PART2_DIR = DRIVE_ROOT / "data_quality_panel" / RUN_ID
assert not PART2_DIR.exists(), f"Run ID already exists: {RUN_ID}"
print("Part 2 run:", RUN_ID)
print("Input:", SOURCE_DIR)
print("New output:", PART2_DIR)
print("Availability assumption (days):", AVAILABILITY_LAG_DAYS)


## 4. Automated quality audit and clean panel construction

The program checks the source partitions in chronological order, writes one typed clean Parquet partition per quarter, aggregates missingness and anomaly counts, and reconciles terminal reporting histories with dated Failure and Institution events. A manifest and validation report are written **only after** all outputs have passed integrity checks. This operation does not run Part 1 again.


In [ ]:
PART2_CMD = [
    sys.executable, str(REPO_DIR / "run_data_quality.py"), "run",
    "--source-dir", str(SOURCE_DIR),
    "--output-dir", str(PART2_DIR),
    "--availability-lag-days", str(AVAILABILITY_LAG_DAYS),
    "--git-commit", COMMIT_SHA,
]
subprocess.run(PART2_CMD, cwd=REPO_DIR, check=True)

import json
with (PART2_DIR / "part2_validation.json").open(encoding="utf-8") as f:
    QUALITY_VALIDATION = json.load(f)
assert QUALITY_VALIDATION["success"] is True
assert QUALITY_VALIDATION["quarters"] == 170
assert QUALITY_VALIDATION["source_rows"] == ACQUISITION["financial_rows"]
assert (QUALITY_VALIDATION["clean_rows"] +
        QUALITY_VALIDATION["quarantined_rows"] == QUALITY_VALIDATION["source_rows"])
print("Source rows:", f"{QUALITY_VALIDATION['source_rows']:,}")
print("Clean panel rows:", f"{QUALITY_VALIDATION['clean_rows']:,}")
print("Quarantined rows:", QUALITY_VALIDATION["quarantined_rows"])
print("Quality pipeline: PASSED")


## 5. Observed data-quality findings

The tables are read directly from the generated evidence files. The missingness table identifies fields requiring later training-only preprocessing; the rule counts distinguish screening flags from excluded observations. Figures summarize field missingness and quarter coverage without presenting hypothetical findings.


In [ ]:
import csv
from IPython.display import display, Markdown, Image, HTML

def read_evidence(name):
    with (PART2_DIR / name).open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def markdown_table(title, columns, records):
    display(Markdown("**" + title + "**"))
    header = " | ".join(columns)
    separator = " | ".join("---" for _ in columns)
    body = "\n".join(" | ".join(str(row.get(c, "")) for c in columns)
                     for row in records)
    display(Markdown(header + "\n" + separator + "\n" + body))

missing = sorted(read_evidence("quality_variables.csv"),
                 key=lambda r: (-float(r["missing_pct"]), r["field"]))
markdown_table("Highest field-level missingness (top 15)",
               ["field", "missing_rows", "missing_pct"], missing[:15])
markdown_table("Screening and quarantine rules",
               ["rule", "flagged_source_rows", "flagged_pct"],
               read_evidence("quality_rules.csv"))
for heading, file in (
    ("Figure 1. Financials field missingness", "field_missingness.png"),
    ("Figure 2. Reporting observations by quarter", "quarterly_coverage.png"),
):
    display(Markdown("**" + heading + "**"))
    display(Image(filename=str(PART2_DIR / "figures" / file), width=740))
    display(HTML('<div style="height:18px"></div>'))


## 6. Terminal events and point-in-time safeguards

`terminal_exits.csv` is a **retrospective audit**, not a feature or label table. Only a dated, CERT-linked FDIC failure is classified as confirmed failure. A dated institution termination without a linked failure is recorded as another exit, not automatically a merger. Unresolved discontinuations, conflicting dates and right-censored banks remain separate. The clean panel includes no `FAILDATE`, `ACTIVE`, `CHANGEC1` or retrospective terminal classification.


In [ ]:
from collections import Counter
import pyarrow.parquet as pq

exits = read_evidence("terminal_exits.csv")
exit_counts = Counter(row["TERMINAL_CLASS"] for row in exits)
markdown_table("Retrospective terminal-record classifications",
               ["class", "banks"],
               [{"class": key, "banks": exit_counts[key]}
                for key in sorted(exit_counts)])
linkage_counts = Counter(row["LINKAGE_STATUS"]
                         for row in read_evidence("failure_linkage.csv"))
markdown_table("Failure-event linkage status",
               ["linkage", "records"],
               [{"linkage": key, "records": linkage_counts[key]}
                for key in sorted(linkage_counts)])
display(Markdown("**Figure 3. Retrospective terminal-record classifications**"))
display(Image(filename=str(PART2_DIR / "figures" / "terminal_exits.png"), width=740))

# Inspect physical columns from one panel file, without pulling all 170 quarters.
partitions = sorted((PART2_DIR / "panel").rglob("part-00000.parquet"))
assert len(partitions) == 170
panel_columns = pq.ParquetFile(partitions[0]).schema_arrow.names
for forbidden in ("FAILDATE", "ACTIVE", "CHANGEC1", "ENDEFYMD",
                  "TERMINAL_CLASS", "future_failure_label"):
    assert forbidden not in panel_columns, f"Retrospective information in panel: {forbidden}"
print("Clean panel columns:", len(panel_columns))
print("Retrospective information in panel: NONE")


## 7. Independent verification and Part 3 handoff

This check rereads source lineage hashes, verifies every panel partition's schema/row count/SHA-256, reconciles clean and quarantined counts, and checks every generated quality evidence file. It does not reprocess the FDIC data. The downstream processing stage must consume the clean Parquet panel and keep retrospective exit evidence out of predictors.


In [ ]:
subprocess.run([
    sys.executable, str(REPO_DIR / "run_data_quality.py"), "verify",
    "--source-dir", str(SOURCE_DIR),
    "--output-dir", str(PART2_DIR),
], cwd=REPO_DIR, check=True)

with (PART2_DIR / "part2_manifest.json").open(encoding="utf-8") as f:
    PART2_MANIFEST = json.load(f)
assert PART2_MANIFEST["git_commit"] == COMMIT_SHA
assert PART2_MANIFEST["exit_classification_is_retrospective_only"] is True
assert PART2_MANIFEST["source_manifest_sha256"] == ACQUISITION["manifest_sha256"]
assert PART2_MANIFEST["source_verification_sha256"] == ACQUISITION["verification_report_sha256"]
assert len(PART2_MANIFEST["partitions"]) == 170
print("Part 2 evidence: PASSED")
print("Run ID:", RUN_ID)
print("Clean panel:", PART2_DIR / "panel")
print("Quality audit:", PART2_DIR / "quality_report.md")
print("Retrospective exit audit:", PART2_DIR / "terminal_exits.csv")
print("Final validation:", PART2_DIR / "part2_validation.json")


## 8. Handoff and limitations

The `panel/` dataset is the sole Part 2 input to Part 3 processing benchmarks. `quality_variables.csv`, `quality_by_year.csv`, `quality_rules.csv`, `quarantined_rows.csv` and figures document the data-quality decisions. `terminal_exits.csv` and `failure_linkage.csv` are held separately for later target-definition work; they **must not** be joined to predictive features as contemporaneous variables.

For Part 4, the four-quarter label for `q` requires a fully observed `(q, q+4 quarters]` outcome window; per the exam, the last eligible report is 2025Q2. The actual FDIC reporting-publication timestamp is not available in Part 1, so the 90-day availability convention remains an assumption to evaluate before modelling.

Save this executed notebook back to `notebooks/03_data_quality_panel_colab.ipynb` on GitHub. Do not commit Parquet, CSV/PNG evidence, or Drive data files.
